In [ ]:
import pandas as pd
import os
import glob

file_paths = glob.glob('data/dataset/*.csv')
all_stocks_data = []

In [ ]:
for file in file_paths:
    
    dataframe = pd.read_csv(file)
    if 'Close' not in dataframe.columns:
        print(f" WARNING: Skipping {file}. It does not have a 'Close' column!")
        continue
    
    dataframe['Name'] = os.path.basename(file).replace('.csv', '')
    
    # 1> Simple moving average (SMA) over 50 days and 20 days
    sma_50 = dataframe['Close'].rolling(window=50).mean()
    sma_20 = dataframe['Close'].rolling(window=20).mean()
    dataframe['SMA_50_pct'] = (dataframe['Close']-sma_50)/sma_50
    dataframe['SMA_20_pct'] = (dataframe['Close']-sma_20)/sma_20

    # 2> Daily Return (Percentage jump/fall from the previous day)
    dataframe['Daily_Return'] = dataframe['Close'].pct_change()
    
    # 3> Relative Strength Index (RSI) for 14 days window
    rsi_window = 14
    delta = dataframe['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=rsi_window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_window).mean()
    rs = gain / loss                                                    # relative strength
    dataframe['RSI'] = 100 - (100 / (1 + rs)) 
    
    # 4> Bollinger Bands (BB) for 20 days window
    bb_window = 20
    bb_mid = dataframe['Close'].rolling(window=bb_window).mean()
    bb_std = dataframe['Close'].rolling(window=bb_window).std()         # standard deviation of close price within the window
    bb_upper = bb_mid + (2 * bb_std)                    # upper limit of bollinger band
    bb_lower = bb_mid - (2 * bb_std)                    # lower limit of bollinger band
    dataframe['BB_Upper_pct'] = (bb_upper-dataframe['Close'])/dataframe['Close']
    dataframe['BB_Lower_pct'] = (dataframe['Close']-bb_lower)/dataframe['Close']

    dataframe['Next_Day_Return'] = dataframe['Close'].pct_change().shift(-1)
    
    features = ['Open','High','Low','Close','Volume','SMA_20_pct', 'SMA_50_pct', 'BB_Upper_pct', 'BB_Lower_pct', 'RSI', 'Daily_Return','Next_Day_Return']
    dataframe = dataframe.dropna(subset=features)
    
    split_index = int(len(dataframe) * 0.8)
    dataframe['Dataset_Tag'] = ['Train' if i < split_index else 'Test' for i in range(len(dataframe))]
    
    all_stocks_data.append(dataframe[['Dataset_Tag','Name'] + features])

In [ ]:
master_df = pd.concat(all_stocks_data, ignore_index=True)
master_df.to_csv('data/universal_training_data.csv', index=False)